# 09 · External Tables and File Formats

> **Module 11 · File Formats · Lectures 11.1**

In this notebook you will:

- Create an external table over files in GCS
- Query external data with and without partitioning
- Use wildcards and the `table_id` pseudo-column
- Load CSV, NDJSON, Avro, Parquet, ORC and TFRecord files
- Decide between an external table and a managed table

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. External vs managed tables

| | Managed | External |
|---|---|---|
| Storage | BigQuery-owned (Colossus) | Your GCS bucket |
| Load cost | $0.02/GiB once | $0 |
| Query cost | Bytes scanned | Bytes scanned (+ GCS egress if outside) |
| Writes | SQL or API | Only your pipeline |
| Partitioning | Native | Hive-style from file layout |
| Performance | Slightly faster | Slightly slower |
| Best for | Owned data | Shared / archive / lakehouse data

In [ ]:
# The bucket the sample data lives in
BUCKET = os.environ.get("COURSE_BUCKET", f"{PROJECT_ID}-bq-course")
STAGING = f"gs://{BUCKET}/landing/"
EXTERNAL_DIR = f"gs://{BUCKET}/external/"

print(f"Bucket: {BUCKET}")
print(f"Landing: {STAGING}")

# Make sure the datasets exist
for ds in ("external", "staging", "raw"):
    dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{ds}")
    dataset_ref.location = "US"
    client.create_dataset(dataset_ref, exists_ok=True)
    print(f"✓ dataset {ds}")

## 2. Create an external table over CSV files

An external table is a **schema over files**. The data stays in GCS; the table
only describes how to read it.

In [ ]:
EXTERNAL_CSV = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.external.orders_csv` (
  order_id      STRING,
  customer_id   STRING,
  order_date    DATE,
  total_amount  NUMERIC,
  status        STRING,
  region        STRING,
  channel       STRING
)
WITH PARTITION OPTIONS (
  format = 'CSV',
  uris = ['{EXTERNAL_DIR}orders/*.csv'],
  skip_leading_rows = 1,
  field_delimiter = ','
)
"""

run_query(EXTERNAL_CSV)
print("✓ External table created")

table = client.get_table(f"{PROJECT_ID}.external.orders_csv")
print(f"  type: {table.table_type}")
print(f"  uri:  {table.external_data_configuration.source_uris}")

In [ ]:
# Query it exactly like a managed table
df = run_to_dataframe(
    f"""
    SELECT order_id, customer_id, total_amount, region
    FROM `{PROJECT_ID}.external.orders_csv`
    LIMIT 10
    """
)
display(df)

## 3. The same thing with the Python client

`ExternalSourceConfiguration` is the SDK equivalent of `WITH PARTITION OPTIONS`.

In [ ]:
ext = bigquery.Table(f"{PROJECT_ID}.external.orders_csv_py")
ext.external_data_configuration = bigquery.ExternalSourceConfiguration(
    autodetect=False,
    source_format=bigquery.SourceFormat.CSV,
    source_uris=[f"{EXTERNAL_DIR}orders/*.csv"],
    options=bigquery.ExternalSourceFormatOptions(
        skip_leading_rows=1,
        field_delimiter=",",
    ),
)
ext.schema = [
    bigquery.SchemaField("order_id", "STRING"),
    bigquery.SchemaField("customer_id", "STRING"),
    bigquery.SchemaField("order_date", "DATE"),
    bigquery.SchemaField("total_amount", "NUMERIC"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("region", "STRING"),
    bigquery.SchemaField("channel", "STRING"),
]
ext = client.create_table(ext, exists_ok=True)
print(f"✓ {ext.full_table_id} ({ext.table_type})")

## 4. Hive-partitioned external tables

When the file layout encodes the partition, BigQuery can prune it. The path
segment **before** the file name is the partition key.

In [ ]:
# Expected layout:
#   gs://BUCKET/external/orders_hive/order_date=2026-01-15/orders-*.csv
#   gs://BUCKET/external/orders_hive/order_date=2026-01-16/orders-*.csv

run_query(
    f"""
    CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.external.orders_hive` (
      order_id      STRING,
      customer_id   STRING,
      total_amount  NUMERIC,
      status        STRING,
      region        STRING,
      channel       STRING
    )
    PARTITION BY order_date
    WITH PARTITION OPTIONS (
      format = 'CSV',
      uris = ['{EXTERNAL_DIR}orders_hive/*/*'],
      skip_leading_rows = 1,
      hive_partitioning_mode = 'AUTO'
    )
    """
)
print("✓ Hive-partitioned external table created")

# The partition column appears as a regular column
df = run_to_dataframe(
    f"""
    SELECT order_date, COUNT(*) AS n, ROUND(SUM(total_amount), 2) AS revenue
    FROM `{PROJECT_ID}.external.orders_hive`
    GROUP BY order_date
    ORDER BY order_date
    """
)
display(df)

In [ ]:
# Pruning on a Hive-partitioned external table
PRUNED_EXT = f"SELECT COUNT(*) FROM `{PROJECT_ID}.external.orders_hive` WHERE order_date = DATE '2026-01-15'"
FULL_EXT = f"SELECT COUNT(*) FROM `{PROJECT_ID}.external.orders_hive`"

for label, sql in (("full", FULL_EXT), ("pruned", PRUNED_EXT)):
    est = estimate_cost(sql)
    print(f"{label:<8} {est['bytes_processed']:>14,} bytes  ${est['estimated_cost_usd']:.6f}")

## 5. Wildcards and the `table_id` pseudo-column

A wildcard table matches many files with one table name. Use `_` (single
character) to match exactly one character and `*` to match any number.

In [ ]:
# A wildcard external table — one table, many files
run_query(
    f"""
    CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.external.orders_wild` (
      order_id      STRING,
      customer_id   STRING,
      order_date    DATE,
      total_amount  NUMERIC,
      status        STRING,
      region        STRING,
      channel       STRING
    )
    WITH PARTITION OPTIONS (
      format = 'CSV',
      uris = ['{EXTERNAL_DIR}orders/*.csv'],
      skip_leading_rows = 1
    )
    """
)
)

# The pseudo-column exposes which file each row came from
df = run_to_dataframe(
    f"""
    SELECT
      REGEXP_EXTRACT(table_id, r'[^/]+$') AS source_file,
      COUNT(*) AS rows_in_file,
      MIN(order_date) AS first_date,
      MAX(order_date) AS last_date
    FROM `{PROJECT_ID}.external.orders_wild`
    GROUP BY source_file
    ORDER BY source_file
    """
)
display(df)

## 6. File formats — the comparison

| Format | Nested | Compressed | Columnar | Schema | Load speed |
|---|---|---|---|---|---|
| CSV | ❌ | ❌ | ❌ | Flexible | Fast |
| NDJSON | ✅ | ❌ | ❌ | Flexible | Fast |
| Avro | ✅ | ❌ | ❌ | In-file | Fast |
| Parquet | ✅ | ✅ | ✅ | In-file | Fastest |
| ORC | ✅ | ✅ | ✅ | In-file | Fast |
| TFRecord | ✅ | ❌ | ❌ | In-file | Fast |

In [ ]:
# ---- Parquet ----
run_query(
    f"""
    CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.external.orders_parquet` (
      order_id      STRING,
      customer_id   STRING,
      order_date    DATE,
      total_amount  NUMERIC,
      status        STRING,
      region        STRING,
      channel       STRING
    )
    WITH PARTITION OPTIONS (
      format = 'PARQUET',
      uris = ['{EXTERNAL_DIR}orders/*.parquet']
    )
    """
)
)
print("✓ Parquet external table")

# ---- NDJSON ----
run_query(
    f"""
    CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.external.orders_ndjson` (
      order_id      STRING,
      customer_id   STRING,
      order_date    DATE,
      total_amount  NUMERIC,
      status        STRING,
      region        STRING,
      channel       STRING
    )
    WITH PARTITION OPTIONS (
      format = 'NEWLINE_DELIMITED_JSON',
      uris = ['{EXTERNAL_DIR}orders/*.ndjson']
    )
    """
)
)
print("✓ NDJSON external table")

# ---- Avro ----
run_query(
    f"""
    CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.external.orders_avro` (
      order_id      STRING,
      customer_id   STRING,
      order_date    DATE,
      total_amount  NUMERIC,
      status        STRING,
      region        STRING,
      channel       STRING
    )
    WITH PARTITION OPTIONS (
      format = 'AVRO',
      uris = ['{EXTERNAL_DIR}orders/*.avro']
    )
    """
)
)
print("✓ Avro external table")

In [ ]:
# Compare the bytes scanned per format for the same logical data
FORMATS = {
    "CSV":    f"SELECT COUNT(*) FROM `{PROJECT_ID}.external.orders_csv`",
    "NDJSON": f"SELECT COUNT(*) FROM `{PROJECT_ID}.external.orders_ndjson`",
    "Avro":   f"SELECT COUNT(*) FROM `{PROJECT_ID}.external.orders_avro`",
    "Parquet": f"SELECT COUNT(*) FROM `{PROJECT_ID}.external.orders_parquet`",
}

print(f"{'format':<10} {'bytes read':>14} {'est. cost':>12}")
print("-" * 40)
for name, sql in FORMATS.items():
    try:
        est = estimate_cost(sql)
        print(f"{name:<10} {est['bytes_processed']:>14,} ${est['estimated_cost_usd']:>11.6f}")
    except Exception as exc:
        print(f"{name:<10} ERROR: {str(exc)[:60]}")

## 7. Column pruning on Parquet

This is the big win: Parquet stores each column separately, so reading one
column reads only that column's bytes.

In [ ]:
PARQUET_ALL = f"SELECT * FROM `{PROJECT_ID}.external.orders_parquet`"
PARQUET_ONE = f"SELECT order_id FROM `{PROJECT_ID}.external.orders_parquet`"
PARQUET_THREE = f"SELECT order_id, order_date, total_amount FROM `{PROJECT_ID}.external.orders_parquet`"

for label, sql in (
    ("SELECT *", PARQUET_ALL),
    ("one column", PARQUET_ONE),
    ("three columns", PARQUET_THREE),
):
    est = estimate_cost(sql)
    print(f"{label:<16} {est['bytes_processed']:>12,} bytes")

print("\nParquet column pruning is automatic — no extra configuration needed.")

## 8. Loading files into a managed table

External tables cost $0 to load. A managed table copies the data into Colossus
and is faster to query, at $0.02/GiB.

In [ ]:
# The one-liner: external table -> managed table
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.orders_from_parquet`
    PARTITION BY order_date
    CLUSTER BY customer_id, region
    AS
    SELECT * FROM `{PROJECT_ID}.external.orders_parquet`
    """
)
print("✓ Managed table created from the external table")

table = client.get_table(f"{PROJECT_ID}.curated.orders_from_parquet")
print(f"  rows:   {table.num_rows:,}")
print(f"  bytes:  {table.num_bytes:,}")
print(f"  partition: {table.time_partitioning.field}")
print(f"  clustering: {table.clustering_fields}")

In [ ]:
# ---- LoadJobConfig: the SDK equivalent of a LOAD statement ----
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema_update_options=[
        bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
        bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION,
    ],
    time_partitioning=bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.DAY, field="order_date"
    ),
    clustering_fields=["customer_id", "region"],
)

job = client.load_table_from_uri(
    f"{EXTERNAL_DIR}orders/*.parquet",
    f"{PROJECT_ID}.curated.orders_loaded_py",
    job_config=job_config,
)
job.result()
print(f"✓ Loaded {job.output_rows:,} rows via load_table_from_uri")

In [ ]:
# Export a managed table back out to Parquet — the reverse direction
EXPORT_CONFIG = bigquery.ExportJobConfig(
    destination_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    use_avro_logical_types=True,
)

job = client.extract_table(
    f"{PROJECT_ID}.curated.orders_from_parquet",
    f"{EXTERNAL_DIR}exported_orders/",
    job_config=EXPORT_CONFIG,
)
job.result()
print("✓ Exported to Parquet")

## 9. BigLake and Iceberg

Iceberg tables in BigQuery give you ACID transactions on open tables, and
BigLake brings the metadata into BigQuery's catalog for managed performance.

In [ ]:
# Reading an Iceberg table in BigLake
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.external.lake_orders`
    WITH CONNECTION
      `us-central1.{PROJECT_ID}.lake.connection`
    OPTIONS (
      format = 'ICEBERG',
      uris = ['{EXTERNAL_DIR}lake/orders'],
      table_format = 'ICEBERG'
    )
    """
)
print("✓ Iceberg table created (needs a BigLake connection)")

print("BigLake + Iceberg gives:")
print("  - ACID transactions and schema evolution")
print("  - Time travel and partition filtering in BigQuery")
print("  - Open-table interoperability with Spark / Trino / Snowflake")

## 10. External table best practices

- **Always partition externally** with a date column — it is the only way to prune
  on an external table.
- Use **Parquet** unless you need a streaming-friendly format; it is smallest and
  prunes columns automatically.
- Use **NDJSON** for semi-structured data you want to explore with JSON functions.
- External tables have **no DML** — `INSERT`/`UPDATE`/`MERGE` are not supported.
- External tables are **not cached**; repeated scans cost the same every time.
- Watch out for **path depth**: `uris` must include the wildcard, and the file
  must be directly under it.
- Keep the bucket and the BigQuery dataset in the **same region** to avoid egress.
- For very large scans, a **managed table** is cheaper than the repeated re-scan.

## 🔑 Takeaways

- An external table is a **schema over files in GCS** — no data is copied.
- **Hive partitioning** (`order_date=2026-01-15/`) is what makes pruning work.
- **Parquet prunes columns automatically** — the biggest external-table win.
- External tables have **no DML** and **no cache**.
- Use `table_id` to trace a row back to its source file.
- Managed table = one-time $0.02/GiB load, then fast scans and full DML support.
- **Iceberg + BigLake** for open-table ACID semantics in BigQuery.

---

## 📚 Next

[10 · Pipeline Patterns](./10-pipeline-patterns.ipynb)